# SynthID-Image-Eval: Example Usage

This notebook demonstrates how to use the SynthID-Image-Eval toolkit for evaluating Google's SynthID watermarking technology.

## Purpose

This is a security research tool designed to:
- Test the robustness of SynthID watermarks
- Identify potential vulnerabilities
- Support responsible disclosure to Google

## Setup

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Imports
from generators.image_generator import ImageGenerator
from transformers.image_transformer import ImageTransformer
from detectors.synthid_detector import SynthIDDetector
from utils.config_loader import get_config
from utils.results_analyzer import ResultsAnalyzer

import matplotlib.pyplot as plt
from PIL import Image

print("✅ Imports successful")

## 1. Image Generation

Generate images using Google's image generation models (which apply SynthID watermarks).

In [ ]:
# Configuration
PROJECT_ID = "your-google-cloud-project-id"
LOCATION = "us-central1"

# Initialize generator
generator = ImageGenerator(
    project_id=PROJECT_ID,
    location=LOCATION,
    model_name="imagegeneration@006",
    output_dir="../data/generated"
)

# Generate test images
test_prompts = [
    "A sunset over mountains",
    "A futuristic city skyline",
    "An abstract art piece with geometric shapes"
]

generated_images = generator.generate_batch(
    prompts=test_prompts,
    images_per_prompt=3
)

print(f"Generated {sum(len(v) for v in generated_images.values())} images")

### Display Generated Images

In [ ]:
# Display some generated images
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sample_images = []
for images in generated_images.values():
    if images:
        sample_images.append(images[0])
    if len(sample_images) >= 3:
        break

for ax, img_path in zip(axes, sample_images):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(img_path.stem[:30])
    ax.axis('off')

plt.tight_layout()
plt.show()

## 2. Image Transformation

Apply various transformations to test watermark robustness.

In [ ]:
# Initialize transformer
transformer = ImageTransformer(output_dir="../data/transformed")

# Define transformations to test
test_transformations = [
    {'type': 'compression', 'quality': 85},
    {'type': 'compression', 'quality': 50},
    {'type': 'resize', 'scale_factor': 0.5},
    {'type': 'rotation', 'angle': 5},
    {'type': 'gaussian_noise', 'intensity': 0.05},
    {'type': 'blur', 'kernel_size': 5},
    {'type': 'crop', 'crop_percentage': 0.1},
    {'type': 'brightness', 'factor': 0.8},
]

# Transform a sample image
sample_image = sample_images[0]
transformed_results = transformer.transform_image(
    sample_image,
    test_transformations
)

print(f"Created {len(transformed_results)} transformed versions")

### Visualize Transformations

In [ ]:
# Display original and transformed images
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.flatten()

# Original
original = Image.open(sample_image)
axes[0].imshow(original)
axes[0].set_title("Original (with SynthID)")
axes[0].axis('off')

# Transformed
for idx, (trans_path, metadata) in enumerate(transformed_results[:8], 1):
    img = Image.open(trans_path)
    transform_type = metadata['transformation']['type']
    axes[idx].imshow(img)
    axes[idx].set_title(transform_type)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 3. SynthID Detection Testing

Test whether Gemini can detect SynthID watermarks in transformed images.

In [ ]:
# Configuration
GEMINI_API_KEY = "your-gemini-api-key"

# Initialize detector
detector = SynthIDDetector(
    api_key=GEMINI_API_KEY,
    model_name="gemini-1.5-pro",
    output_dir="../results"
)

# Test baseline (original image)
print("Testing baseline detection...")
baseline_result = detector.detect_single_image(sample_image)
print(f"Baseline: AI-generated={baseline_result.get('is_ai_generated')}, "
      f"Confidence={baseline_result.get('confidence')}")

# Test transformed images
print("\nTesting transformed images...")
transformed_paths = [path for path, _ in transformed_results]
transformed_detection = detector.detect_batch(transformed_paths[:5])  # Test first 5

# Summary
detected_count = sum(1 for r in transformed_detection if r.get('is_ai_generated') == True)
print(f"\nDetected {detected_count}/{len(transformed_detection)} transformed images as AI-generated")

### Analyze Detection Results

In [ ]:
import pandas as pd

# Create results DataFrame
results_data = []
for result in transformed_detection:
    img_name = Path(result['image_path']).stem
    results_data.append({
        'Image': img_name[:40],
        'Detected as AI': result.get('is_ai_generated', 'Unknown'),
        'Confidence': result.get('confidence', 'N/A'),
        'Has SynthID': result.get('has_synthid_markers', 'Unknown')
    })

df = pd.DataFrame(results_data)
display(df)

## 4. Results Visualization

Analyze and visualize the effectiveness of different transformations.

In [ ]:
# Initialize analyzer
analyzer = ResultsAnalyzer(results_dir="../results")

# Create comparison
baseline_results = [baseline_result]
summary = analyzer.create_summary_statistics(baseline_results, transformed_detection)

display(summary)

### Plot Detection Rates

In [ ]:
analyzer.plot_detection_rates(
    baseline_results,
    transformed_detection,
    output_file=Path("../results/detection_comparison.png")
)

# Display the plot
img = Image.open("../results/detection_comparison.png")
plt.figure(figsize=(12, 6))
plt.imshow(img)
plt.axis('off')
plt.show()

## 5. Full Pipeline Example

Running the complete evaluation pipeline.

In [ ]:
from main import SynthIDEvaluationPipeline

# Initialize pipeline with config
pipeline = SynthIDEvaluationPipeline(config_path="../config/config.yaml")

# Run full pipeline
# Note: This will take significant time and API calls
# pipeline.run_full_pipeline()

## 6. Custom Analysis

Perform custom analysis on your results.

In [ ]:
# Load saved results
import json

# Example: Load and analyze specific transformation types
# with open('../results/detection_results_TIMESTAMP.json', 'r') as f:
#     all_results = json.load(f)

# Analyze which transformations are most effective at evading detection
# ...


## Next Steps

1. Run experiments with different transformation parameters
2. Test combination transformations (e.g., compression + rotation)
3. Analyze results and identify vulnerabilities
4. Document findings for responsible disclosure to Google

## Responsible Disclosure

Any vulnerabilities discovered should be reported through Google's Vulnerability Reward Program:
https://bughunters.google.com/